In [ ]:
pip install tsp_solver2

In [ ]:
from map import Map, Marker, Layout
from ipyleaflet import AntPath
import db
from tsp_solver.greedy import solve_tsp

In [ ]:
libraries = db.query(
    """
with visited as (
select 'Harold Washington Library Center' as branch union
select 'Manning'
)
select l.branch
, st_x(l.geom) as lat
, st_y(l.geom) as lon
, st_distance(ll.geom, l.geom) as hw_distance
, v.branch is not null as vistited

from stg_chicago__libraries l
left join stg_chicago__libraries ll on ll.branch = 'Harold Washington Library Center'
left join visited v on l.branch = v.branch

order by hw_distance asc
"""
)
libraries

In [ ]:
m = Map((41.85, -87.5))
m.m.layout = Layout(height="1000px")

path = []
for row in libraries.fetchall():
    m.markers.append(
        Marker(
            (row[1], row[2]),
            f"{row[1]}",
            "check" if row[4] else "",
            color=("green" if row[4] else "blue"),
        )
    )
    path.append([row[1], row[2]])

dist = lambda p1, p2: ((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2) ** 0.5
distance_matrix = [
    [dist(path[i], path[j]) for j in range(len(path))] for i in range(len(path))
]
m.add(AntPath(locations=[path[i] for i in solve_tsp(distance_matrix)], delay=5000))

m

In [ ]:
# csv log template
# print("\n".join([','.join([l[0], '', '']) for l in libraries.fetchall()]))